In [34]:
print(1)

1


In [35]:
import os
import time
import uuid
import json
from dotenv import load_dotenv

from portkey_ai import Portkey ,createHeaders , PORTKEY_GATEWAY_URL


load_dotenv()


PORTKEY_API_KEY = os.getenv("PORTKEY_API_KEY", "toHeU5iBWu0hDC+8A/NHbKqfAfcy")

portkey = Portkey(api_key=PORTKEY_API_KEY)


In [41]:
GROQ_SLUG = "gkey"
GROQ_MODEL = f"@{GROQ_SLUG}/openai/gpt-oss-120b"


GROQ_SLUG_2 = "gkey"
GROQ_MODEL_SMALL = f"@{GROQ_SLUG_2}/openai/gpt-oss-20b"


# Keep GROQ_API_KEY for the Baseline experiment (direct Groq call — no gateway)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")


In [37]:
print("Setup complete!")
print(f"  Portkey API Key : {'OK' if PORTKEY_API_KEY else 'MISSING'}")
print(f"  Groq slug       : {GROQ_SLUG}")
print(f"  Groq model ref  : {GROQ_MODEL}")
print(f"  Groq slug 2     : {GROQ_SLUG_2}")
print(f"  Small model ref : {GROQ_MODEL_SMALL}")
print(f"\nPortkey Gateway : {PORTKEY_GATEWAY_URL}")
print(f"\nGroq api-key : {GROQ_API_KEY}")

Setup complete!
  Portkey API Key : OK
  Groq slug       : gkey
  Groq model ref  : @gkey/gpt-oss-120b
  Groq slug 2     : gkey
  Small model ref : @gkey/gpt-oss-20b

Portkey Gateway : https://api.portkey.ai/v1

Groq api-key : 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


In [38]:
def section(title):
    print(f"\n{'='*62}")
    print(f"  {title}")
    print(f"{'='*62}")

def show(q, answer, ms, label=""):
    bar = chr(9472) * 62
    print(f"\n{bar}")
    print(f"Q: {q}")
    print(f"A: {answer[:260]}{'...' if len(answer) > 260 else ''}")
    note = f" | {label}" if label else ""
    print(f"⏱  {ms:.0f}ms{note}")
    print(bar)


### Raw LLM Calls

In [39]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

raw_groq = ChatGroq(api_key=GROQ_API_KEY, model="openai/gpt-oss-120b", temperature=0)

section("BASELINE — Direct Groq Call")

questions = [
    "What is Bitcoin scripts?",
    "What is Intel SRIOV?",
]

for q in questions:
    t0 = time.time()
    r = raw_groq.invoke([HumanMessage(content=q)])
    show(q, r.content, (time.time()-t0)*1000, label="direct Groq — no gateway")



  BASELINE — Direct Groq Call

──────────────────────────────────────────────────────────────
Q: What is Bitcoin scripts?
A: **Bitcoin Script – a quick‑look guide**

---

## 1. What is it?

- **Bitcoin Script** is the tiny, stack‑based programming language that lives inside every Bitcoin transaction.
- It is **not a general‑purpose language** (you can’t write loops or complex contra...
⏱  12759ms | direct Groq — no gateway
──────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────
Q: What is Intel SRIOV?
A: **Intel SR‑IOV (Single Root I/O Virtualization)** is a hardware‑level technology that lets a single physical network (or other I/O) device appear as multiple independent virtual devices. Each of those virtual devices—called **Virtual Functions (VFs)**—can be a...
⏱  6697ms | direct Groq — no gateway
──────────────────────────────────────────────────────────────


1st Portkey Experiment

In [42]:
section("EXP 1 — Basic Gateway Call")

questions = [
    "What is AI and gen ai ?",
    "What is coffee and black coffee?",
]


for q in questions:
    t0 = time.time()
    r = portkey.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role":"user","content":q}]
    )
    show(q, r.choices[0].message.content, (time.time()-t0)*1000,
         label="routed via Portkey gateway")

print("\n✅ Check portkey.ai → Logs to see both requests fully logged!")
print("   Token count, cost, latency — all tracked. Zero extra code.")


  EXP 1 — Basic Gateway Call

──────────────────────────────────────────────────────────────
Q: What is AI and gen ai ?
A: **Artificial Intelligence (AI)**  

- **Definition:** AI is the field of computer science that builds systems capable of performing tasks that normally require human intelligence. These tasks include learning, reasoning, perception, language understanding, dec...
⏱  4040ms | routed via Portkey gateway
──────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────
Q: What is coffee and black coffee?
A: **Coffee – a quick overview**

| Aspect | What it means |
|--------|---------------|
| **Basic definition** | A brewed beverage made from the roasted seeds (often called “beans”) of the coffee plant *Coffea* (most commonly *Coffea arabica* or *Coffea canephora...
⏱  3688ms | routed via Portkey gateway
──────────────────────────────────────────────────────────────

✅ Check portkey.ai → Logs to see both re